In [3]:
import numpy as np
import pandas as pd
import country_converter as coco

In [4]:
# Initialize converter
cc = coco.CountryConverter()

# Load UN speeches dataset
speeches = pd.read_parquet("../data/un_speeches.parquet")
# Load Conflict dyad dataset
conflict_dyads = pd.read_excel("../data/2026_ucdp-brd-dyadic-261.xlsx")

# 2. Convert GWNo codes (gwno_a) to standard 3-letter ISO codes
# (This handles historical changes and maps directly to your UN speech country codes)
conflict_dyads['iso3_a'] = cc.pandas_convert(series=conflict_dyads['gwno_a'], src='GWcode', to='ISO3')
conflict_dyads['iso3_b'] = cc.pandas_convert(series=conflict_dyads['gwno_b'], src='GWcode', to='ISO3')

# Ensure merge key data types are string/object across all DataFrames
conflict_dyads['iso3_a'] = conflict_dyads['iso3_a'].astype(str)
conflict_dyads['year'] = conflict_dyads['year'].astype(int)

speeches['country_code'] = speeches['country_code'].astype(str)
speeches['year'] = speeches['year'].astype(int)

# 3. Join UCDP with your UN Speeches DataFrame (df)
# Merges speech data of Side A (usually the state/government) in a given conflict year
ucdp_with_speeches = pd.merge(
    conflict_dyads,
    speeches,
    left_on=['iso3_a', 'year'],
    right_on=['country_code', 'year'],
    how='left'
)

678 not found in GWcode
345 not found in GWcode
900, 200, 2 not found in GWcode
200, 2 not found in GWcode
200, 2 not found in GWcode
678 not found in GWcode


In [6]:
# 4. Join with Ideal Point Dyads (if adding ideological distance)
# Load your pre-calculated dyadic ideal points
ideal_dyads = pd.read_csv("../data/IdealPointDyads1946-2025.csv")

# Standardize type matching for the dyadic join
ideal_dyads['year'] = ideal_dyads['year'].astype(int)

# Merge dyadic ideal points matching Side A and Side B of the UCDP conflict
final_merged = pd.merge(
    ucdp_with_speeches,
    ideal_dyads,
    left_on=['gwno_a', 'gwno_b', 'year'],
    right_on=['ccode1', 'ccode2', 'year'],
    how='left'
)

print("Merge complete! Final dataset shape:", final_merged.shape)

Merge complete! Final dataset shape: (2081, 39)


In [7]:
final_merged[~final_merged['AbsIdealDiff'].isnull()]

,conflict_id,dyad_id,location_inc,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,...,filename,ccode1,iso3c1,Countryname1,ccode2,iso3c2,Countryname2,IdealPointFP1,IdealPointFP2,AbsIdealDiff
42,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_44_1989.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.753585,-0.791073,0.037487
43,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_45_1990.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.786355,-0.480959,0.305396
44,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_46_1991.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.625393,-0.844177,0.218784
45,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_47_1992.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.689053,-0.936180,0.247128
46,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_49_1994.txt,750.0,IND,India,770.0,PAK,Pakistan,-1.364695,-0.951268,0.413426
47,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_51_1996.txt,750.0,IND,India,770.0,PAK,Pakistan,-1.677913,-1.032368,0.645545
48,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_52_1997.txt,750.0,IND,India,770.0,PAK,Pakistan,-1.078334,-0.863602,0.214732
49,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_53_1998.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.637275,-0.640085,0.002810
50,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_54_1999.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.555591,-0.539317,0.016274
51,218,422,"India, Pakistan",Government of India,141,NaN,Government of Pakistan,142,NaN,1,...,IND_55_2000.txt,750.0,IND,India,770.0,PAK,Pakistan,-0.606680,-0.844924,0.238244


iso3c1      object
year         int64
text        object
filename    object
dtype: object

In [8]:
conflict_dyads

,conflict_id,dyad_id,location_inc,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,...,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,gwno_battle,region,version,iso3_a,iso3_b
0,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,630,2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
1,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
3,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
4,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2076,309,18621,Sudan,Government of Sudan,112,NaN,SFA,9645,NaN,2,...,625,NaN,NaN,NaN,625,625,4,26.1,SDN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2077,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2078,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2079,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
